# Project GoldenHour-RL: Unsloth 4-bit QLoRA + GRPO Training Notebook
### Verifiable Emergency Crisis Dispatch Optimization Engine

This notebook trains `Qwen/Qwen2.5-Coder-1.5B-Instruct` using **Group Relative Policy Optimization (GRPO)** and **Unsloth** on a single free NVIDIA T4 GPU (Kaggle or Google Colab).

**Hardware Target:** 1x T4 (16GB VRAM)  
**Expected Peak VRAM:** ~8.4 GB (~45% reduction vs PPO)  
**Training Time:** ~1.5 - 2 hours (150 steps)

## 1. Verify GPU Hardware

In [ ]:
!nvidia-smi

## 2. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets wandb pytest python-dotenv

## 3. Clone Repository

In [ ]:
!git clone https://github.com/akshaykumarhudedmani/goldenhour-rl.git
%cd goldenhour-rl

## 4. Set Environment Credentials (W&B and Hugging Face)

In [ ]:
import os

# Set your credentials (you can also use Kaggle/Colab Secrets)
os.environ["WANDB_API_KEY"] = "your_wandb_api_key_here"
os.environ["HF_TOKEN"] = "your_hf_token_here"
os.environ["WANDB_PROJECT"] = "goldenhour-rl"

!wandb login --relogin $WANDB_API_KEY

## 5. Verify Deterministic Sandbox Unit Tests

In [ ]:
!pytest tests/ -v

## 6. Generate Synthetic Datasets

In [ ]:
# Generate 300 verifiable training samples and 100 evaluation samples
!python -m goldenhour_rl.data.crisis_generator --output data/crisis_train.jsonl --num_samples 300 --seed 42
!python -m goldenhour_rl.data.crisis_generator --output data/crisis_eval.jsonl --num_samples 100 --seed 999

## 7. Run Baseline Heuristic Telemetry Benchmark

In [ ]:
!python -m goldenhour_rl.eval.benchmark --eval_data data/crisis_eval.jsonl --dry_run

## 8. Run GRPO Reinforcement Learning Training Loop

In [ ]:
!python -m goldenhour_rl.training.train_grpo \
    --model_name "Qwen/Qwen2.5-Coder-1.5B-Instruct" \
    --data_path "data/crisis_train.jsonl" \
    --output_dir "outputs/goldenhour_grpo" \
    --max_steps 150 \
    --use_wandb

## 9. Push Trained LoRA Weights to Hugging Face Hub

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "akshaykumarhudedmani/goldenhour-rl"

api.create_repo(repo_id=repo_id, exist_ok=True, token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path="outputs/goldenhour_grpo/final_lora",
    repo_id=repo_id,
    token=os.environ["HF_TOKEN"],
)
print(f"Successfully uploaded trained LoRA weights to https://huggingface.co/{repo_id}")